HUANG Flora <br>
- Langage : Python <br>
- Système d'exploitation : Windows -> sur Jupyter <br>
- URL de notre fichier d'intérêt: https://www.data.gouv.fr/datasets/localisation-des-pharmacies-dans-openstreetmap <br>
- mise à jour du fichier (pharmacies_points): 09/06/26 <br>
- Date de finalisation du script : **25/06/26** (permet le recalcul Les dates estimées sur les fichiers CSV)

# PARTIE 1 : Création liste des pharmacies de la France métropolitaine 

**Objectif** : scraping de tous les commentaires des pharmacies de la France métropolitaine (hors outre-mer) <br>

**Remarque** : le script de la partie II nécessite l'ouverture de vraie fenêtre visible pour le scraping -> `headless=False` (dans notre cas, il permet de vérifier en temps réel si le scraping a bien été effectué) -> donc pas possible d'utiliser Google Colab 


In [3]:
# Installation à faire avant le lancé du projet 
# !pip install pyproj 
# !pip install playwright 
# playwright install chromium

In [4]:
import pandas as pd
liste_pharma = pd.read_csv("pharmacies_point.csv", low_memory=False)
liste_pharma.head(3)

,FID,osm_id,amenity,name,short_name,official_name,alt_name,old_name,operator,operator-type,...,wikipedia,description,opening_hours,source,note,osm_version,osm_timestamp,the_geom,osm_original_geom,osm_type
0,pharmacies_point.4534591198,4534591198,pharmacy,Pharmacie Centrale de Bondy,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Mo-Sa 08:30-19:30,Le ministère des solidarités et de la santé - ...,NaN,7,2024-05-11T16:57:42Z,POINT (276175.8531618072 6260215.629555009),NaN,node
1,pharmacies_point.172156934,172156934,pharmacy,Pharmacie des Andaines,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,cadastre-dgi-fr source : Direction Générale de...,NaN,11,2023-08-18T08:56:31Z,POINT (-40562.0055358363 6204977.067562205),SRID=3857;POLYGON((-40572.837448406004 6204977...,way
2,pharmacies_point.3918927589,3918927589,pharmacy,Pharmacie Cayeux Etaploise,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"Mo-Fr 08:30-12:30,14:00-19:30; Sa 08:30-12:30,...",Celtipharm - 10/2014,NaN,6,2025-10-09T20:46:23Z,POINT (182438.6302862845 6535708.511748197),NaN,node


In [2]:
liste_pharma.shape

(19137, 44)

In [3]:
liste_pharma.dtypes

FID                   object
osm_id                 int64
amenity               object
name                  object
short_name            object
official_name         object
alt_name              object
old_name              object
operator              object
operator-type         object
dispensing            object
emergency             object
capacity             float64
wheelchair            object
social_facility      float64
ref-FR-FINESS         object
type-FR-FINESS        object
ref-FR-NAF            object
ref-FR-SIRET          object
website               object
contact-website       object
url                   object
phone                 object
contact-phone         object
fax                   object
contact-fax           object
email                 object
contact-email         object
addr-housename        object
addr-housenumber      object
addr-street           object
addr-city             object
addr-postcode         object
wikidata              object
wikipedia     

Compréhension des variables à travers les sites et différents recherches : 
- https://wiki.openstreetmap.org/wiki/FR:Page_principale <br>
- https://wiki.openstreetmap.org/wiki/FR:%C3%89l%C3%A9ments_cartographiques <br>
- https://wiki.openstreetmap.org/wiki/FR:Tag:amenity%3Dpharmacy <br>
- https://taginfo.geofabrik.de/europe:france/


Variables : 
- **FID** : L'identifiant unique du point sur OpenStreetMap (pour éviter les doublons). 
- osm_id : Le contributeur qui a créé/modifié le point et la date de mise à jour. 
- amenity : //
- **name** : Le nom de la pharmacie 
- short_name : //
- official_name : //
- alt_name  : //
- old_name : //
- operator : //
- operator-type : // 
- dispensing : si la pharmacie peut délivrer des médicament sur ordonnance ou non (yes / no) 
- emergency : 
- capacity : 
- wheelchair : L'accessibilité en fauteuil roulant (yes / no) 
- social_facility : 
- **ref-FR-FINESS** : Le code FINESS national 
- type-FR-FINESS : 
- ref-FR-NAF : 
- **ref-FR-SIRET** : Le numéro SIRET de l'entreprise 
- website :  // 
- contact-website :  // 
- url :  // 
- phone : 
- contact-phone : 
- fax :  // 
- contact-fax :  // 
- email :  // 
- contact-email :  // 
- addr-housename :  // 
- addr-housenumber :  Le numéro dans la rue  
- addr-street : Le nom de la rue 
- addr-city : La commune 
- **addr-postcode** : Le code postal 
- wikidata :  // 
- wikipedia :  // 
- description : 
- opening_hours : Les horaires d'ouverture
- source :  // 
- note : 
- osm_version :  // 
- osm_timestamp :  // 
- **the_geom** : coordonnée en longitude/latitude ou X/Y
- osm_original_geom :  // 
- osm_type :  // 


In [4]:
# Sélection des variables pertinantes 
variables_pertinants = ['FID', 'ref-FR-FINESS', 'ref-FR-SIRET','name','addr-city', 'addr-postcode','the_geom']

# Filtrage de la liste pour ne garder que les colonnes qui existent dans notre tableau
colonnes_presentes = [col for col in variables_pertinants if col in liste_pharma.columns]

df_analyse = liste_pharma[colonnes_presentes].copy()

print(df_analyse.shape)
print(df_analyse.head())

(19137, 7)
                            FID ref-FR-FINESS    ref-FR-SIRET  \
0   pharmacies_point.4534591198     930006820             NaN   
1    pharmacies_point.172156934     610786485             NaN   
2   pharmacies_point.3918927589     620010942  83493738500013   
3   pharmacies_point.5101170481     570016774             NaN   
4  pharmacies_point.11429216164     610785909  84099976700014   

                          name addr-city addr-postcode  \
0  Pharmacie Centrale de Bondy     Bondy         93140   
1       Pharmacie des Andaines       NaN           NaN   
2   Pharmacie Cayeux Etaploise   Étaples         62630   
3           Pharmacie Lorraine       NaN           NaN   
4        Pharmacie de la Halle       NaN           NaN   

                                      the_geom  
0  POINT (276175.8531618072 6260215.629555009)  
1  POINT (-40562.0055358363 6204977.067562205)  
2  POINT (182438.6302862845 6535708.511748197)  
3  POINT (767638.8259116503 6306264.502831527)  
4  P

In [5]:
df_analyse.isnull().sum()

FID                  0
ref-FR-FINESS     1541
ref-FR-SIRET     10960
name              1265
addr-city        14988
addr-postcode    14758
the_geom             0
dtype: int64

Pour la variable "the_geom", on suit la méthode de Nedah -> on sépare les coordonnées de Point(lon, lat) en 2 colonnes : X(longitude) et Y(latitude) <br>
URL : https://github.com/nedahh1/analyse-avis-pharmarcies-grand-est 

In [6]:
# Nettoyage et séparation 
nettoye = df_analyse['the_geom'].str.upper().str.replace('POINT (', '', regex=False).str.replace(')', '', regex=False)
df_analyse[['X_mercator', 'Y_mercator']] = nettoye.str.split(' ', expand=True).astype(float)

# On supprime l'ancienne colonne geom
df_analyse = df_analyse.drop(columns=['the_geom'])

# Filtre France métropolitaine adapté au Web Mercator (en incluant valeurs négatifs (ouest))
df_metro = df_analyse[
    (df_analyse['X_mercator'].between(-600000, 1200000)) &  
    (df_analyse['Y_mercator'].between(5000000, 6700000))
].copy()

df_metro.shape

(19137, 8)

Ici on applique un filtrage géographique (= Bounding Box ou boîte de délimitation) : on ne garde que les lignes où la latitude et la longitude sont comprises entre les valeurs données (ci dessus). 
Cela correspond géographiquement au rectangle de la France métropolitaine et éliminer les DOM-TOM (outre-mer) <br>
URL : https://wiki.openstreetmap.org/wiki/Bounding_box

**Web Mercator** : projection cartographique de Mercator utilisée pour les coordonnées dans le système WGS 84 Web Mercatorou WGS 84/Pseudo-Mercator , ainsi que pour la visualisation. <br>
URL : https://en.wikipedia.org/wiki/Web_Mercator_projection#:~:text=However%2C%20the%20Web%20Mercator%20uses,maps%20at%20the%20same%20scale.

Signification des valeurs : coordonnées de distance géographique -> unité appliquée = mètre (m).
- X(-600000, 1200000) : distance par rapport au Méridien de Greenwich (la ligne verticale qui coupe la France vers Bordeaux/Caen)
    - -600 000 mètres (-600 km) : C'est la limite Ouest du rectangle. On va jusqu'à 600 km à l'ouest du méridien pour englober toute la Bretagne (Brest) et le Pays Basque. 
    - 1 200 000 mètres (+1 200 km) : C'est la limite Est. On va jusqu'à 1 200 km à l'est du méridien pour englober Strasbourg, Nice et toute la Corse.

- Y(5000000, 6700000) : axe mesure la distance en ligne droite depuis l'Équateur (la ligne imaginaire au milieu de la Terre)
    - 5 000 000 mètres (5 000 km) : C'est la limite Sud de notre rectangle. La Corse et Perpignan se trouvent à un peu plus de 5 100 km au nord de l'équateur. En démarrant à 5 000 km, on s'assure de ne rien rater au Sud.
    - 6 700 000 mètres (6 700 km) : C'est la limite Nord. Dunkerque se trouve à environ 6 620 km au nord de l'équateur. La barre à 6 700 km permet d'englober tout le nord de la France.
 
Plus d'info concernant la géographie de la France : https://fr.wikipedia.org/wiki/G%C3%A9ographie_de_la_France 

In [7]:
df_metro.head()

,FID,ref-FR-FINESS,ref-FR-SIRET,name,addr-city,addr-postcode,X_mercator,Y_mercator
0,pharmacies_point.4534591198,930006820,NaN,Pharmacie Centrale de Bondy,Bondy,93140,276175.853162,6.260216e+06
1,pharmacies_point.172156934,610786485,NaN,Pharmacie des Andaines,NaN,NaN,-40562.005536,6.204977e+06
2,pharmacies_point.3918927589,620010942,83493738500013,Pharmacie Cayeux Etaploise,Étaples,62630,182438.630286,6.535709e+06
3,pharmacies_point.5101170481,570016774,NaN,Pharmacie Lorraine,NaN,NaN,767638.825912,6.306265e+06
4,pharmacies_point.11429216164,610785909,84099976700014,Pharmacie de la Halle,NaN,NaN,69996.838651,6.234936e+06


A partir de nos coordonées en mètre, on essaye de déterminer les BAN (base adresse nationale) des différents pharmacies via le site adresse.data.gouv (site qui permet une convertion automatique des paramètres de coordonnée -> dans notre cas, des coordonnées en mètre en une locatlisation GPS et autres...)<br>
URL : https://adresse.data.gouv.fr/outils 

In [8]:
#!pip install pyproj 

Ici on va faire appel au site à travers un code pour faire une conversion de nos coordonnées mètres (X/Y) en degrés GPS (lon/lat),
soit EPSG:3857 (Web Mercator) -> EPSG:4326 (GPS standard attendu par la BAN)

**Librairie io** :  pour créer un fichier virtuel temporaire en RAM pour envoyer les données sans encombrer le disque dur. <br>
**Librairie requests** :  pour se connecter à internet pour envoyer les coordonnées et recevoir les réponses de la BAN. <br>
**Librairie pyproj** : pour effectuer des transformations cartographiques et des calculs géodésiques. <br>
**Fonction Transformer** : pour convertir des coordonnées d'un système de cartographie à un autre.

In [9]:
import io 
import requests 
from pyproj import Transformer 

In [10]:
transformer = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
df_metro['lon'], df_metro['lat'] = transformer.transform(df_metro['X_mercator'].values, df_metro['Y_mercator'].values)

print("en cours d'envoi...")

# On fait appel à l'API de la BAN 
url_ban = "https://api-adresse.data.gouv.fr/reverse/csv/"
csv_en_texte = df_metro.to_csv(index=False).encode('utf-8')
reponse = requests.post(url_ban, files={'data': ('pharmacies_metro.csv', csv_en_texte)})

if reponse.status_code == 200:
    df_final = pd.read_csv(io.StringIO(reponse.text))
    df_final.to_csv("pharmacies_metropole_final.csv", index=False, sep=";")
    print(" nouveau fichier a été créé.")
else:
    print(f"Erreur de communication avec la BAN. Code erreur : {reponse.status_code}")

en cours d'envoi...
 nouveau fichier a été créé.


status_code :  code de réponse que renvoie n'importe quel site internet quand on l'interroge. <br>
nombre 200 : code universel sur le web pour dire "Succès".

In [11]:
df_final.isnull().sum()

FID                       0
ref-FR-FINESS          1541
ref-FR-SIRET          10960
name                   1265
addr-city             14988
addr-postcode         14758
X_mercator                0
Y_mercator                0
lon                       0
lat                       0
result_longitude      19137
result_latitude       19137
result_distance          54
result_label             54
result_type              54
result_id                54
result_banId           2207
result_housenumber      570
result_name              54
result_street           570
result_postcode          54
result_city              54
result_context           54
result_citycode          54
result_oldcitycode    18061
result_oldcity        18054
result_district       17736
result_status             0
dtype: int64

In [12]:
df_final.head()

,FID,ref-FR-FINESS,ref-FR-SIRET,name,addr-city,addr-postcode,X_mercator,Y_mercator,lon,lat,...,result_name,result_street,result_postcode,result_city,result_context,result_citycode,result_oldcitycode,result_oldcity,result_district,result_status
0,pharmacies_point.4534591198,930006820,NaN,Pharmacie Centrale de Bondy,Bondy,93140,276175.853162,6.260216e+06,2.480930,48.913611,...,26 Avenue Suzanne Buisson,Avenue Suzanne Buisson,93140.0,Bondy,"93, Seine-Saint-Denis, Île-de-France",93010,NaN,NaN,NaN,ok
1,pharmacies_point.172156934,610786485,NaN,Pharmacie des Andaines,NaN,NaN,-40562.005536,6.204977e+06,-0.364375,48.586434,...,53 Rue Félix Desaunay,Rue Félix Desaunay,61600.0,La Ferté Macé,"61, Orne, Normandie",61168,61168.0,La Ferté-Macé,NaN,ok
2,pharmacies_point.3918927589,620010942,83493738500013,Pharmacie Cayeux Etaploise,Étaples,62630,182438.630286,6.535709e+06,1.638874,50.513637,...,32 Place du Général de Gaulle,Place du Général de Gaulle,62630.0,Étaples,"62, Pas-de-Calais, Hauts-de-France",62318,NaN,NaN,NaN,ok
3,pharmacies_point.5101170481,570016774,NaN,Pharmacie Lorraine,NaN,NaN,767638.825912,6.306265e+06,6.895817,49.184730,...,Rue Nationale,NaN,57600.0,Forbach,"57, Moselle, Grand Est",57227,NaN,NaN,NaN,ok
4,pharmacies_point.11429216164,610785909,84099976700014,Pharmacie de la Halle,NaN,NaN,69996.838651,6.234936e+06,0.628792,48.764147,...,19 Rue Saint-jean,Rue Saint-jean,61300.0,L'Aigle,"61, Orne, Normandie",61214,NaN,NaN,NaN,ok


In [13]:
df_final.dtypes

FID                    object
ref-FR-FINESS          object
ref-FR-SIRET           object
name                   object
addr-city              object
addr-postcode          object
X_mercator            float64
Y_mercator            float64
lon                   float64
lat                   float64
result_longitude      float64
result_latitude       float64
result_distance       float64
result_label           object
result_type            object
result_id              object
result_banId           object
result_housenumber     object
result_name            object
result_street          object
result_postcode       float64
result_city            object
result_context         object
result_citycode        object
result_oldcitycode    float64
result_oldcity         object
result_district        object
result_status          object
dtype: object

In [14]:
# Sélection des variables pertinantes 
variables_pertinants2 = ['FID','name','lon', 'lat', 'result_label', 'result_city', 'result_context']

# Filtrage de la liste pour ne garder que les colonnes qui existent dans notre tableau
colonnes_presentes2 = [col for col in variables_pertinants2 if col in df_final.columns]

df_propre = df_final[colonnes_presentes2].copy()

df_propre.head

<bound method NDFrame.head of                                 FID                         name       lon  \
0       pharmacies_point.4534591198  Pharmacie Centrale de Bondy  2.480930   
1        pharmacies_point.172156934       Pharmacie des Andaines -0.364375   
2       pharmacies_point.3918927589   Pharmacie Cayeux Etaploise  1.638874   
3       pharmacies_point.5101170481           Pharmacie Lorraine  6.895817   
4      pharmacies_point.11429216164        Pharmacie de la Halle  0.628792   
...                             ...                          ...       ...   
19132   pharmacies_point.1008156074            Pharmacie Breuzin  0.698130   
19133  pharmacies_point.13179599050   Pharmacie de la République  3.035218   
19134   pharmacies_point.3712079795         Pharmacie des Arènes  6.187351   
19135   pharmacies_point.1933342224            Pharmacie Jacques  6.226526   
19136   pharmacies_point.1152714441               Pharmacie Mery  0.694787   

             lat                 

In [15]:
df_propre.isnull().sum()

FID                  0
name              1265
lon                  0
lat                  0
result_label        54
result_city         54
result_context      54
dtype: int64

- 1265 pharmacies dont les noms ne sont pas connus
- 54 pharmacies dont les adresses ne sont pas retrouvées après la conversion 

## Vérification
Comparaison du nb de pharmacie par région via notre fiche -> avec la carte régionale des officines de la France par région du site suivant : <br>
URL : https://www.ordre.pharmacien.fr/je-suis/pharmacien/je-suis-pharmacien-titulaire-d-officine/mon-exercice-professionnel/les-cartes <br>
mise à jour 05/06/26.

In [16]:
df_propre['region'] = df_propre['result_context'].str.split(',').str[2].str.strip()

In [17]:
df_propre['region'].value_counts()

region
Île-de-France                 3303
Auvergne-Rhône-Alpes          2353
Nouvelle-Aquitaine            1995
Occitanie                     1904
Hauts-de-France               1749
Provence-Alpes-Côte d'Azur    1705
Grand Est                     1468
Pays de la Loire              1029
Bretagne                       957
Bourgogne-Franche-Comté        955
Normandie                      851
Centre-Val de Loire            702
Corse                          112
Name: count, dtype: int64

In [18]:
df_propre.to_csv("datalistpharma.csv", index=False, sep=";")

Comparaison : 
| regions                        | officine  "OpenstreetMap" | officine "Ordre.pharmacie" |
|--------------------------------|---------------------------|----------------------------|
| Mise à jour                    | 09/06/26                  | 05/06/26                   |
| Ile-de-Fr                      | 3303                      | 3351                       |
| **Auvergne-Rhône-Alpes**       | 2353                      | 2300                       |
| Nouvelle-Aquitaine             | 1995                      | 1973                       |
| Occitanie                      | 1904                      | 1877                       |
| Hautes-de-Fr                   | 1749                      | 1866                       |
| Provence-Alpes-Côte d'Azur     | 1705                      | 1793                       |
| Grand Est                      | 1468                      | 1545                       |
| Pays de la Loire               | 1029                      | 1025                       |
| **Bretagne**                   | 957                       | 947                        |
| **Bourgogne-Franche-Comté**    | 955                       | 869                        |
| Normandie                      | 851                       | 880                        |
| **Centre-val de Loire**        | 702                       | 726                        |
| **Corse**                      | 112                       | 127                        |
| Total                          | 19137                     | 19279                      |


Outil de construction : https://www.tablesgenerator.com/markdown_tables <br>
Remarque : Dans les travaux de Nedah : 1368 officines dans le Grand Est <br>
Dans notre cas avec le fichier mise à jour le 09/06/26 : 1468 offines dans le Grand Est

# Extraction des avis et des réponses 

Ce qui peut nous intéresser : 
- ID/Nom et adresse de la pharmacie.
- Note de l'avis (étoiles).
- Texte de l'avis + Date de publication de l'avis.
- Texte de la réponse de la pharmacie + Date de publication de la réponse (pour calculer le temps de réaction).


Mode d'emploi Playwright : https://playwright.dev/python/docs/intro <br> - librarie plus optimale pour une automatisation de la navigation <br> 
- cependant possible aussi avec Selenium 

## PART I : prépa des fichiers csv par région

In [19]:
import os # sert d'interface entre notre code et le système d'exploitation
import unicodedata # permet de manipuler les caractères Unicode

In [20]:
fichier_source = "datalistpharma.csv"
dossier_sortie = "pharmacies_par_region"

os.makedirs(dossier_sortie, exist_ok=True) # permet la création du dossier : pharmacies_par_region

In [21]:
df = pd.read_csv(fichier_source, sep=";")

regions_metropole = [
    "Auvergne-Rhône-Alpes", "Bourgogne-Franche-Comté", "Bretagne",
    "Centre-Val de Loire", "Corse", "Grand Est", "Hauts-de-France",
    "Île-de-France", "Normandie", "Nouvelle-Aquitaine", "Occitanie",
    "Pays de la Loire", "Provence-Alpes-Côte d'Azur"]

# Nettoyage des noms de région 
def nom_fichier_safe(region):
    # Supprime les accents
    sans_accents = unicodedata.normalize('NFD', region).encode('ascii', 'ignore').decode('utf-8')
    # Remplace espaces et apostrophes par des underscores + minuscules
    return sans_accents.lower().replace(" ", "_").replace("'", "_").replace("-", "_")

print("découpage par région en cours...")

nom_colonne = "region"

# Boucle de filtrage 
for region in df[nom_colonne].unique():
    if region in regions_metropole:

        df_region = df[df[nom_colonne] == region]

        nom_safe = nom_fichier_safe(region)
        chemin_final = os.path.join(dossier_sortie, f"pharmacies_{nom_safe}.csv")

        df_region.to_csv(chemin_final, index=False, sep=";", encoding="utf-8-sig")
        print(f"Fichier créé : {chemin_final} ({len(df_region)} lignes)")

print("traitement terminé.")

découpage par région en cours...
Fichier créé : pharmacies_par_region\pharmacies_ile_de_france.csv (3303 lignes)
Fichier créé : pharmacies_par_region\pharmacies_normandie.csv (851 lignes)
Fichier créé : pharmacies_par_region\pharmacies_hauts_de_france.csv (1749 lignes)
Fichier créé : pharmacies_par_region\pharmacies_grand_est.csv (1468 lignes)
Fichier créé : pharmacies_par_region\pharmacies_auvergne_rhone_alpes.csv (2353 lignes)
Fichier créé : pharmacies_par_region\pharmacies_pays_de_la_loire.csv (1029 lignes)
Fichier créé : pharmacies_par_region\pharmacies_occitanie.csv (1904 lignes)
Fichier créé : pharmacies_par_region\pharmacies_centre_val_de_loire.csv (702 lignes)
Fichier créé : pharmacies_par_region\pharmacies_provence_alpes_cote_d_azur.csv (1705 lignes)
Fichier créé : pharmacies_par_region\pharmacies_nouvelle_aquitaine.csv (1995 lignes)
Fichier créé : pharmacies_par_region\pharmacies_bourgogne_franche_comte.csv (955 lignes)
Fichier créé : pharmacies_par_region\pharmacies_bretagne

## PART II : Scraping

Afin d'éviter un trop long temps, nous répartissons le scraping de la manière suivante : 


| Flora                   | Louise         | Doriane                    |
|-------------------------|----------------|----------------------------|
| Auvergne Rhône Alpes    | Grand Est      | Nouvelle Aquitaine         |
| Bourgogne-Franche-Comté | Haut de France | Occitanie                  |
| Centre-Val de Loire     | Ile de France  | Pays de la Loire           |
| Bretagne                | Normandie      | Provence-Alpes-Côte d'Azur |
| Corse                   | -              | -                          |

Déroulement du scraping... pour commencer : 
- scraping de 3 communes de notre 1er région : Auvergne-Rhône-Alpes
> pour observer si le rendu est bien fait, si oui
- scraping à l'échelle de la 1er région
- scraping des 4 autres régions 

Application d'un mode asynchrone pour réduire l'intervalle de recherche et d'un temps de pause aléatoires entre chaque recherche de pharmacies pour éviter de se faire bloquer.

In [5]:
#!pip install playwright 
# dans terminal : playwright install

## Scraping pour Google Maps 

### Libraires 

In [23]:
import asyncio # exécute plusieurs tâches en même temps
import random  # générer du hasard 
import re # : Sert à chercher, extraire ou modifier du texte de façon très précise
import urllib.parse # Sert à découper, analyser ou formater les adresses de sites internet (URL)

from datetime import datetime, timedelta # Sert à manipuler facilement les dates, les heures, et à calculer des durées
from playwright.async_api import async_playwright

### Fonctions hors Playwright
- pause aléatoire
- conversion date relative en estimée
- construction URL de GoogleMap pour une pharmacie

In [24]:
# Pause aleatoire entre chaque pharmacie (anti-blocage)
async def pause():
    duree = random.uniform(2.0, 5.0)
    await asyncio.sleep(duree)

In [25]:
# Conversion d'une date relative Google en date estimee (AAAA-MM-JJ) dès ajrd

def convertir_date_relative(texte, reference=None):
    if not texte:
        return None

    reference = reference or datetime.now()
    texte = texte.lower().strip()

    if "aujourd" in texte:
        return reference.date().isoformat()
    if "hier" in texte:
        return (reference - timedelta(days=1)).date().isoformat()

    # remplace les nombres ecrits en lettres (un/une) par "1"
    texte = re.sub(r"\b(un|une)\b", "1", texte)

    match = re.search(r"(\d+)\s*(an|annee|mois|semaine|jour|heure|minute)", texte)
    if not match:
        return None

    nombre = int(match.group(1))
    unite = match.group(2)

    if unite in ("an", "annee"):
        delta = timedelta(days=365 * nombre)
    elif unite == "mois":
        delta = timedelta(days=30 * nombre)
    elif unite == "semaine":
        delta = timedelta(weeks=nombre)
    elif unite == "jour":
        delta = timedelta(days=nombre)
    elif unite == "heure":
        delta = timedelta(hours=nombre)
    else:
        delta = timedelta(minutes=nombre)

    return (reference - delta).date().isoformat()

In [26]:
# Construction de l'URL de recherche Google Maps

def construire_url_maps(nom, adresse):
    requete = f"{nom} {adresse}"
    requete_encodee = urllib.parse.quote(requete)
    return f"https://www.google.com/maps/search/{requete_encodee}"

### Nagivation sur une page 
- gestion du popup de cookies
- gestion de la page de resultats multiples (quand plrs etablissements correspondent a la recherche)
- ouverture de l'onglet "avis"


In [27]:
# Fermeture du cookues de consentement
async def accepter_cookies(page):
    textes_possibles = [
        "Tout accepter",
        "Tout refuser",
        "Accepter tout",
        "Accept all",
        "Reject all",
    ]

    for texte in textes_possibles:
        try:
            bouton = page.get_by_role("button", name=texte) # cherhce sur la page internet un bouton qui porte le texte en cours de test.
            if await bouton.count() > 0:
                await bouton.first.click(timeout=3000) # 3s pour clic avt abondon
                await page.wait_for_timeout(1000) # 1s de temps de pause pour laisse bannière disparaitre
                return True
        except Exception:
            continue

    return False

In [28]:
# Clic sur un resultat de la liste de pharmacie 

async def ouvrir_premier_resultat_si_liste(page, nom_recherche=None, max_tentatives=4):
    try:
        for tentative in range(max_tentatives):
            resultats = page.locator("a.hfpxzc") # id CSS pour lien pharmacie
            nb_resultats = await resultats.count()

            if nb_resultats == 0:
                return tentative > 0

            indice_choisi = 0

            # si on connait le nom recherche, on essaie de trouver le 
            # resultat dont le nom correspond le mieux
            if nom_recherche:
                nom_recherche_normalise = nom_recherche.strip().lower()
                for i in range(min(nb_resultats, 10)):
                    try:
                        aria = await resultats.nth(i).get_attribute("aria-label", timeout=1500)
                        if aria and nom_recherche_normalise in aria.strip().lower():
                            indice_choisi = i
                            break
                    except Exception:
                        continue

            await resultats.nth(indice_choisi).click(timeout=5000)
            await page.wait_for_timeout(1800)

            # on verifie si on est arrive sur une vraie fiche : presence
            # des tabs Presentation/Avis/A propos ET plus aucun lien
            # a.hfpxzcdans panneau principal 
            nb_tabs = await page.get_by_role("tab").count()
            nb_liens_restants = await page.locator("div[role='main'] a.hfpxzc").count()

            if nb_tabs >= 3 and nb_liens_restants == 0:
                return True

        return True
    except Exception:
        pass
    return False

In [29]:
# Clic sur le bouton "Avis" de la fiche pharmacie (si present)

async def ouvrir_onglet_avis(page):
    try:
        bouton_avis = page.get_by_role("tab", name=re.compile("avis", re.IGNORECASE))
        await bouton_avis.first.click(timeout=5000)
        return True
    except Exception:
        return False

### Identification du bon panneau et fermeture de modale 

Le code suivant va permettre de forcer le bot à se concentrer uniquement sur la vraie fiche de la pharmacie (en ignorant les listes de résultats qui restent cachées en arrière-plan) et il refermera automatiquement la pop-up de partage si le bot clique dessus par erreur

In [30]:
# Identification du bon div[role='main'] : celui qui correspond a
# la fiche de l'etablissement (aria-label non vide)

async def trouver_panneau_fiche(page):
    panneaux = page.locator("div[role='main']")
    nb_panneaux = await panneaux.count()

    for i in range(nb_panneaux):
        try:
            aria = await panneaux.nth(i).get_attribute("aria-label", timeout=1500)
            nb_liens = await panneaux.nth(i).locator("a.hfpxzc").count()
            if aria and nb_liens == 0:
                return panneaux.nth(i)
        except Exception:
            continue

    # si aucun panneau ne correspond pas exactement au critere,
    # on prend le premier qui n'a pas de liens residuels, ou a defaut
    # le tout premier panneau trouve
    for i in range(nb_panneaux):
        try:
            nb_liens = await panneaux.nth(i).locator("a.hfpxzc").count()
            if nb_liens == 0:
                return panneaux.nth(i)
        except Exception:
            continue

    return panneaux.first

In [31]:
# Fermeture de la fenetre "Partager" si elle s'est ouverte par accident 
async def fermer_modale_partage_si_presente(page):
    try:
        # clic uniquement sur le champ "Lien a partager" avec l'URL maps.app.goo.gl
        champ_lien = page.locator("text=/maps\\.app\\.goo\\.gl/")
        if await champ_lien.count() == 0:
            return False

        # confirmation supplementaire : le bouton "Copier le lien"
        # n'existe que dans cette modale precise
        bouton_copier = page.get_by_text("Copier le lien", exact=False)
        if await bouton_copier.count() == 0:
            return False

        # fermeture via la croix (X) de la modale
        bouton_fermer = page.locator("button[aria-label*='Fermer'], button[aria-label*='Close']")
        if await bouton_fermer.count() > 0:
            await bouton_fermer.first.click(timeout=2000)
            await page.wait_for_timeout(500)
            return True

        return False
    except Exception:
        return False

### Scroll des avis

In [32]:
# Scroll de la liste des avis jusqu'a chargement complet
# scroll de souris = fonction "mouse.wheel"

async def scroller_avis(page, pause_min=0.8, pause_max=1.5, max_tentatives_sans_evolution=10, debug=False):
    nb_avis_precedent = -1
    tentatives_sans_evolution = 0
    await fermer_modale_partage_si_presente(page)
    
    # On utilise trouver_panneau_fiche plutot que .first : une liste
    # de desambiguisation residuelle peut rester presente avec un
    # aria-label vide, et .first capturerait alors le mauvais panneau.
    try:
        panneau = await trouver_panneau_fiche(page)
        boite = await panneau.bounding_box(timeout=3000)
    except Exception:
        boite = None

    if boite:
        # position de scroll : vers le bas du panneau plutot que le
        # centre -> bas du panneau correspond generalement a
        # la zone des avis deja charges
        x_scroll = boite["x"] + boite["width"] / 2
        y_scroll = boite["y"] + boite["height"] - 100
    else:
        x_scroll, y_scroll = 300, 550

    # clic de "réveil" sur le texte d'un avis existant. 
    # Ça évite de cliquer par erreur sur les petits boutons de filtres 
    # (comme "ordonnance" ou "sourire") qui réduiraient la liste des avis 
    x_reveil, y_reveil = None, None
    try:
        zone_texte_avis = page.locator("div[data-review-id] .wiI7pd").first
        if await zone_texte_avis.count() > 0:
            boite_texte = await zone_texte_avis.bounding_box(timeout=2000)
            if boite_texte:
                x_reveil = boite_texte["x"] + min(boite_texte["width"] / 2, 100)
                y_reveil = boite_texte["y"] + boite_texte["height"] / 2
    except Exception:
        pass

    if x_reveil is None:
        x_reveil, y_reveil = x_scroll, y_scroll

        # securite : on verifie que ce point de repli ne tombe pas
        # sur un lien ou un filtre cliquable
        try:
            element_sous_clic = await page.evaluate(f"""
                () => {{
                    const el = document.elementFromPoint({x_reveil}, {y_reveil});
                    if (!el) return null;
                    const lien = el.closest('a.hfpxzc');
                    const bouton = el.closest('button');
                    return (lien || bouton) ? true : false;
                }}
            """)
            if element_sous_clic and boite:
                y_reveil = boite["y"] + 15
        except Exception:
            pass

    # clic de "reveil" avant de commencer le scroll : sur certaines
    # fiches, le panneau a besoin d'un premier clic pour activer le
    # chargement dynamique des avis, avant meme que le scroll ne
    # fasse effet.
    try:
        await page.mouse.click(x_reveil, y_reveil)
        await page.wait_for_timeout(1000)
    except Exception:
        pass

    if debug:
        nb_apres_clic = await page.locator("div[data-review-id]").count()
        print(f"  [debug] apres clic de reveil ({x_reveil:.0f},{y_reveil:.0f}) : {nb_apres_clic} avis")

    # le clic de reveil peut, sur certaines fiches, tomber pile sur
    # un avis deja charge et notamment sur son bouton de partage :
    # on verifie et ferme systematiquement la modale qui en resulterait
    modale_fermee = await fermer_modale_partage_si_presente(page)
    if debug and modale_fermee:
        print("  [debug] une modale de partage a ete fermee apres le clic de reveil")

    while tentatives_sans_evolution < max_tentatives_sans_evolution:
        try:
            for num_geste in range(6):
                await page.mouse.move(x_scroll, y_scroll)
                await page.mouse.wheel(0, 800)
                await asyncio.sleep(0.25)
                if debug:
                    nb_intermediaire = await page.locator("div[data-review-id]").count()
                    print(f"    [debug] apres geste {num_geste+1}/6 : {nb_intermediaire} avis")
        except Exception as e:
            if debug:
                print(f"  [debug] erreur pendant le scroll : {e}")

        modale_fermee = await fermer_modale_partage_si_presente(page)
        if debug and modale_fermee:
            print("  [debug] une modale de partage a ete fermee pendant le scroll")

        await asyncio.sleep(random.uniform(pause_min, pause_max))

        nb_avis_actuel = await page.locator("div[data-review-id]").count()

        if debug:
            print(f"  [debug] cycle (tentatives sans evolution={tentatives_sans_evolution}) : {nb_avis_actuel} avis")

        if nb_avis_actuel <= nb_avis_precedent:
            tentatives_sans_evolution += 1
        else:
            tentatives_sans_evolution = 0

        nb_avis_precedent = nb_avis_actuel

In [33]:
# Clic sur "Plus" si Avis non complet 

async def deplier_texte(bloc_avis, bouton_texte="Plus"):
    try:
        bouton_plus = bloc_avis.get_by_role("button", name=bouton_texte)
        if await bouton_plus.count() > 0:
            await bouton_plus.first.click(timeout=2000)
    except Exception:
        pass

### Extraction des données d'un avis

In [34]:
# Extraction des infos d'un seul bloc d'avis sur Google Maps

async def extraire_un_avis(bloc_avis, reference_date):
    resultat = {
        "review_id": None,
        "note_avis": None,
        "texte_avis": None,
        "date_avis_texte": None,
        "date_avis_estimee": None,
        "texte_reponse": None,
        "date_reponse_texte": None,
        "date_reponse_estimee": None,
    }

    try:
        resultat["review_id"] = await bloc_avis.get_attribute("data-review-id", timeout=2000)
    except Exception:
        pass

    # deplier le texte de l'avis si tronque
    await deplier_texte(bloc_avis, "Plus")

    # note en etoiles : aria-label du type "5 etoiles"
    try:
        bloc_note = bloc_avis.locator("[aria-label*='étoile'], [aria-label*='star']").first
        aria = await bloc_note.get_attribute("aria-label", timeout=2000)
        if aria:
            match_note = re.search(r"(\d+)", aria)
            if match_note:
                resultat["note_avis"] = int(match_note.group(1))
    except Exception:
        pass

    # texte de l'avis
    try:
        texte_avis = await bloc_avis.locator(".wiI7pd").first.inner_text(timeout=2000)
        resultat["texte_avis"] = texte_avis.strip()
    except Exception:
        pass

    # date de publication de l'avis
    try:
        date_avis = await bloc_avis.locator(".rsqaWe").first.inner_text(timeout=2000)
        resultat["date_avis_texte"] = date_avis.strip()
        resultat["date_avis_estimee"] = convertir_date_relative(date_avis, reference_date)
    except Exception:
        pass

    # reponse de la pharmacie (texte + date), bloc distinct s'il existe
    try:
        bloc_reponse = bloc_avis.locator(".CDe7pd").first
        if await bloc_reponse.count() > 0:

            # deplier le texte de la reponse si tronque (bouton "Plus" propre a ce bloc)
            await deplier_texte(bloc_reponse, "Plus")

            # texte complet de la reponse apres depliage
            try:
                texte_reponse = await bloc_reponse.locator(".wiI7pd").first.inner_text(timeout=2000)
                resultat["texte_reponse"] = texte_reponse.strip()
            except Exception:
                pass

            # date de la reponse : Google Maps utilise .DZSIDd dans le bloc reponse,
            # pas .rsqaWe comme pour l'avis — on essaie plusieurs selecteurs connus
            for selecteur_date in [".DZSIDd", ".rsqaWe", "[class*='date']"]:
                try:
                    date_reponse = await bloc_reponse.locator(selecteur_date).first.inner_text(timeout=1500)
                    if date_reponse and date_reponse.strip():
                        resultat["date_reponse_texte"] = date_reponse.strip()
                        resultat["date_reponse_estimee"] = convertir_date_relative(date_reponse, reference_date)
                        break
                except Exception:
                    continue
    except Exception:
        pass

    return resultat

### Scraping complet d'une pharmacie
Combine toutes les etapes precedentes pour une pharmacie : navigation, ouverture des avis, scroll, extraction, avec un retry automatique si le resultat semble incomplet. 

In [35]:
# Scraping complet d'une pharmacie : recherche + tous ses avis

async def _tenter_scraper_pharmacie(navigateur, nom, adresse, url, reference_date, pharmacy_id):
    page = await navigateur.new_page()
    avis_collectes = []

    for tentative in range(2):
        try:
            await page.goto(url, timeout=30000)
            # on attend que le reseau soit reellement stable avant de
            # continuer, plutot qu'un delai fixe : sous charge parallele
            # (plusieurs pages simultanees)
            try:
                await page.wait_for_load_state("networkidle", timeout=10000)
            except Exception:
                pass
            break
        except Exception as erreur_reseau:
            if tentative == 0:
                print(f"Nouvelle tentative pour {nom} apres erreur reseau : {erreur_reseau}")
                await asyncio.sleep(5)
            else:
                print(f"Echec definitif sur {nom} (FID {pharmacy_id}) : {erreur_reseau}")
                await page.close()
                return avis_collectes

    try:
        await page.wait_for_timeout(2000)

        await accepter_cookies(page)
        await page.wait_for_timeout(1000)

        await ouvrir_premier_resultat_si_liste(page, nom_recherche=nom)
        await page.wait_for_timeout(1000)

        avis_ouverts = await ouvrir_onglet_avis(page)
        if not avis_ouverts:
            await page.wait_for_timeout(2000)
            avis_ouverts = await ouvrir_onglet_avis(page)

        if not avis_ouverts:
            await page.close()
            return avis_collectes

        await page.wait_for_timeout(1500)

        for _ in range(8):
            nb_blocs_initial = await page.locator("div[data-review-id]").count()
            if nb_blocs_initial > 0:
                break
            try:
                panneau = await trouver_panneau_fiche(page)
                boite = await panneau.bounding_box(timeout=2000)
                if boite:
                    x_init = boite["x"] + boite["width"] / 2
                    y_init = boite["y"] + boite["height"] / 2
                    for _ in range(4):
                        await page.mouse.move(x_init, y_init)
                        await page.mouse.wheel(0, 800)
                        await asyncio.sleep(0.25)
            except Exception:
                pass
            await page.wait_for_timeout(1500)

        await scroller_avis(page)

        blocs_avis = page.locator("div[data-review-id]")
        nb_avis = await blocs_avis.count()

        ids_deja_vus = set()

        for i in range(nb_avis):
            bloc = blocs_avis.nth(i)
            infos_avis = await extraire_un_avis(bloc, reference_date)

            review_id = infos_avis.get("review_id")
            if review_id is not None:
                if review_id in ids_deja_vus:
                    continue
                ids_deja_vus.add(review_id)

            avis_collectes.append({
                "FID": pharmacy_id,
                "nom_pharmacie": nom,
                "adresse_pharmacie": adresse,
                **infos_avis,
            })

    except Exception as erreur:
        print(f"Erreur sur la pharmacie {nom} (FID {pharmacy_id}) : {erreur}")

    finally:
        await page.close()

    return avis_collectes


async def scraper_une_pharmacie(navigateur, ligne_pharmacie, seuil_suspect=10):
    pharmacy_id = ligne_pharmacie.get("FID")
    nom = str(ligne_pharmacie.get("name") or "")
    adresse = str(ligne_pharmacie.get("result_label") or "")

    url = construire_url_maps(nom, adresse)
    reference_date = datetime.now()

    avis_collectes = await _tenter_scraper_pharmacie(navigateur, nom, adresse, url, reference_date, pharmacy_id)

    # si le resultat est exactement au seuil suspect (ici 10) = avis suspect > blocage? 
    # on retente une fois sur une session fraiche avant d'accepter ce resultat comme definitif.
    if len(avis_collectes) == seuil_suspect:
        print(f"  Resultat suspect ({seuil_suspect} avis) pour {nom}, nouvelle tentative...")
        await asyncio.sleep(3)
        nouvelle_tentative = await _tenter_scraper_pharmacie(navigateur, nom, adresse, url, reference_date, pharmacy_id)

        # on garde le meilleur des deux resultats (le plus grand nombre d'avis)
        if len(nouvelle_tentative) > len(avis_collectes):
            avis_collectes = nouvelle_tentative

    return avis_collectes

### Scraping en parallèle de plusieurs pharmacies

In [36]:
# Scraping avec N pages en parallele + sauvegarde 

async def scraper_lot_pharmacies(df_pharmacies, chemin_csv_sortie, nb_pages_parallele=5, mode_ajout=False):
    # securite : on retire les pharmacies sans nom renseigne (NaN ou
    # vide) : car imprecise et risque de scraper un etablissement different
    nb_avant_filtre = len(df_pharmacies)
    df_pharmacies = df_pharmacies[df_pharmacies["name"].notna() & (df_pharmacies["name"].str.strip() != "")].copy()
    nb_ignorees_sans_nom = nb_avant_filtre - len(df_pharmacies)
    if nb_ignorees_sans_nom > 0:
        print(f"{nb_ignorees_sans_nom} pharmacie(s) ignoree(s) car sans nom renseigne.")

    # securite : on retire les doublons de pharmacies en entree
    # (meme nom + meme adresse), pour eviter de scraper deux fois
    df_pharmacies = df_pharmacies.drop_duplicates(subset=["name", "result_label"]).copy()

    # mode_ajout=False : on demarre un nouveau fichier (ecrase l'existant)
    # mode_ajout=True  : on ajoute a la suite d'un fichier deja existant
    #                     (utile pour traiter une region commune par commune)
    fichier_existe_deja = mode_ajout and os.path.exists(chemin_csv_sortie)

    async with async_playwright() as playwright:
        # Remarque : headless=True est detecte et bloque par Google Maps
        # headless=False est la seule configuration fiable confirmee par nos tests.
        navigateur = await playwright.chromium.launch(
            headless=False,
            args=["--no-sandbox"],
        )

        semaphore = asyncio.Semaphore(nb_pages_parallele)

        # utilisation d'une liste pour pouvoir modifier la valeur
        # depuis l'interieur de traiter_une_pharmacie (closure)
        fichier_existe_deja_liste = [fichier_existe_deja]

        async def traiter_une_pharmacie(ligne):
            async with semaphore:
                avis = await scraper_une_pharmacie(navigateur, ligne)
                await pause()

                print(f"Pharmacie {ligne['name']} : {len(avis)} avis recuperes")

                if avis:
                    df_avis = pd.DataFrame(avis)
                    df_avis.to_csv(
                        chemin_csv_sortie,
                        mode="a" if fichier_existe_deja_liste[0] else "w",
                        header=not fichier_existe_deja_liste[0],
                        index=False,
                        sep=";",
                        encoding="utf-8-sig",
                    )
                    fichier_existe_deja_liste[0] = True

                return avis

        taches = [traiter_une_pharmacie(ligne) for _, ligne in df_pharmacies.iterrows()]

        # gather attend que TOUTES les taches soient terminees
        # (avec succes ou erreur) avant de continuer, contrairement
        # a as_completed qui peut laisser des taches en suspens
        # si la boucle est interrompue avant leur tour
        await asyncio.gather(*taches, return_exceptions=True)

        await navigateur.close()

    print(f"Lot traite. Fichier sauvegarde : {chemin_csv_sortie}")

### TEST : 3 communes d'Auvergne-Rhone-Alpes

In [37]:
# df_region_ara = pd.read_csv(
#     "pharmacies_par_region/pharmacies_auvergne_rhone_alpes.csv",
#     sep=";"
# )

# # selection de 3 communes au hasard pour le test
# communes_test = df_region_ara["result_city"].dropna().unique()[:3]
# df_test = df_region_ara[df_region_ara["result_city"].isin(communes_test)].copy()

# print(f"Communes testees : {list(communes_test)}")
# print(f"Nombre de pharmacies dans le test : {len(df_test)}")

# await scraper_lot_pharmacies(
#     df_test,
#     chemin_csv_sortie="avis_test_3_communes.csv",
#     nb_pages_parallele=4,
# )

In [6]:
# # Vérification présence du fichier csv 
# print(os.path.exists("avis_test_3_communes.csv"))
# df_resultat = pd.read_csv("avis_test_3_communes.csv", sep=";", encoding="utf-8-sig")
# print(df_resultat.shape)
# df_resultat.head()

In [39]:
# print("Pharmacies distinctes :", df_resultat["nom_pharmacie"].nunique())
# print("Avis distincts (review_id) :", df_resultat["review_id"].nunique())
# print("Avec note :", df_resultat["note_avis"].notna().sum())
# print("Avec texte avis :", df_resultat["texte_avis"].notna().sum())
# print("Avec réponse :", df_resultat["texte_reponse"].notna().sum())

## Scraping d'une region
Avec sous découpage des grosses communes et reprise automatique en cas de coupure ou plantage du navigateur 

In [40]:
def decouper_en_sous_lots(df, taille_max=25):
    """Decoupe un DataFrame en sous-lots de taille_max lignes maximum.
    Pour les grosses communes (Lyon, Paris...), cela limite la duree
    de vie de chaque session navigateur et reduit le risque de crash."""
    sous_lots = []
    for debut in range(0, len(df), taille_max):
        sous_lots.append(df.iloc[debut:debut + taille_max])
    return sous_lots


async def scraper_une_region(fichier_region, fichier_sortie, nb_pages_parallele=5, taille_sous_lot=25):
    df_region = pd.read_csv(fichier_region, sep=";")

    # si le fichier de sortie existe deja (suite a une execution
    # precedente ou une coupure), on recupere les adresses deja
    # traitees pour ne pas refaire le travail
    adresses_deja_faites = set()
    if os.path.exists(fichier_sortie):
        df_existant = pd.read_csv(fichier_sortie, sep=";", encoding="utf-8-sig")
        adresses_deja_faites = set(df_existant["adresse_pharmacie"].dropna().unique())
        print(f"Reprise : {len(adresses_deja_faites)} pharmacies deja traitees, elles seront ignorees.")

    communes = df_region["result_city"].dropna().unique()
    print(f"Region : {fichier_region}")
    print(f"Nombre total de communes : {len(communes)}")

    for indice, commune in enumerate(communes, start=1):

        df_commune = df_region[df_region["result_city"] == commune].copy()
        df_commune = df_commune[~df_commune["result_label"].isin(adresses_deja_faites)]

        if df_commune.empty:
            continue

        print(f"[{indice}/{len(communes)}] Commune : {commune} ({len(df_commune)} pharmacies)")

        # decoupage en sous-lots pour les grosses communes 
        sous_lots = decouper_en_sous_lots(df_commune, taille_sous_lot)

        for num_sous_lot, sous_lot in enumerate(sous_lots, start=1):
            if len(sous_lots) > 1:
                print(f"  Sous-lot {num_sous_lot}/{len(sous_lots)} ({len(sous_lot)} pharmacies)")

            # on retente une fois en cas de plantage du navigateur
            # (TargetClosedError ou autre erreur de session)
            for tentative in range(2):
                try:
                    await scraper_lot_pharmacies(
                        sous_lot,
                        chemin_csv_sortie=fichier_sortie,
                        nb_pages_parallele=nb_pages_parallele,
                        mode_ajout=True,
                    )
                    break
                except Exception as erreur_navigateur:
                    print(f"  Erreur navigateur sur ce sous-lot : {erreur_navigateur}")
                    if tentative == 0:
                        print("  Nouvelle tentative pour ce sous-lot...")
                        await asyncio.sleep(5)
                    else:
                        print("  Sous-lot abandonne apres 2 tentatives, on continue avec le suivant.")

    print(f"Scraping termine pour : {fichier_region}")

## Région 1 : Auvergne-Rhone-Alpes
Estimation du scraping avec un PC portable pour cette région : ~25H 

In [41]:
# await scraper_une_region(
#     fichier_region="pharmacies_par_region/pharmacies_auvergne_rhone_alpes.csv",
#     fichier_sortie="avis_auvergne_rhone_alpes.csv",
#     nb_pages_parallele=4,
# )

In [42]:
# # Vérification du fichier csv 
# print(os.path.exists("avis_auvergne_rhone_alpes.csv"))
# df_resultat = pd.read_csv("avis_auvergne_rhone_alpes.csv", sep=";", encoding="utf-8-sig")
# print(df_resultat.shape)
# df_resultat.head()

In [43]:
# print("Pharmacies distinctes :", df_resultat["nom_pharmacie"].nunique())
# print("Avis distincts (review_id) :", df_resultat["review_id"].nunique())
# print("Avec note :", df_resultat["note_avis"].notna().sum())
# print("Avec texte avis :", df_resultat["texte_avis"].notna().sum())
# print("Avec réponse :", df_resultat["texte_reponse"].notna().sum())

In [44]:
# # Verification : nombre de pharmacies a 0 avis et a 10 avis

# def verifier_couverture(fichier_region, fichier_sortie, seuil_suspect=10):
#     df_region = pd.read_csv(fichier_region, sep=";")

#     if not os.path.exists(fichier_sortie):
#         print("Fichier de sortie introuvable.")
#         return

#     df_avis = pd.read_csv(fichier_sortie, sep=";", encoding="utf-8-sig")

#     # nombre d'avis recuperes par pharmacie (cle = adresse)
#     compte_avis = df_avis.groupby("adresse_pharmacie").size()

#     total_pharmacies = df_region["result_label"].nunique()
#     adresses_traitees = set(df_avis["adresse_pharmacie"].dropna().unique())
#     adresses_a_zero = set(df_region["result_label"]) - adresses_traitees
#     adresses_a_dix = set(compte_avis[compte_avis == seuil_suspect].index)
#     adresses_ok = adresses_traitees - adresses_a_dix

#     print(f"Nombre total de pharmacies dans la region : {total_pharmacies}")
#     print(f"Pharmacies a 0 avis (jamais ecrites)        : {len(adresses_a_zero)}")
#     print(f"Pharmacies a exactement {seuil_suspect} avis (suspectes) : {len(adresses_a_dix)}")
#     print(f"Pharmacies correctement scrapees (>10 avis ou <10 reel) : {len(adresses_ok)}")
#     print(f"Total avis collectes dans le fichier          : {len(df_avis)}")

# verifier_couverture(
#     fichier_region="pharmacies_par_region/pharmacies_auvergne_rhone_alpes.csv",
#     fichier_sortie="avis_auvergne_rhone_alpes.csv",
#     seuil_suspect=10,
# )

### Scraping rattrapage
Estimation scraping ~ 6H pour PC portable

> **Cellule de rattrapge à exécuter uniquement pour celle de la région Auvergne (car correction déjà effectuée sur les codes desfonctions précédentes concernant les avis suspects (10) ou 0 avis)** <br>
> avant la correction -> on retrouvait  ~400/2353 pharmacies pour la région Auvergne avec des avis = 0 ou suspects

Remarque : on suppose que des avis = 0 sont dus 
- au nom de la pharmacie qui ne correspond plus au nom actuel
- pharmacies fermées
- pb de scoll à cause des adresses donnant une liste multiple de pharma

Concernant les avis suspects (10) : 
- pb de scoll

In [2]:
# Code rattrapage pour Auvergne

# def identifier_pharmacies_a_reprendre(fichier_region, fichier_sortie, fichier_rattrapage=None, seuil_suspect=10):
#     df_region = pd.read_csv(fichier_region, sep=";")

#     if not os.path.exists(fichier_sortie):
#         print("Aucun fichier de sortie trouve, rien a rattraper.")
#         return df_region.iloc[0:0]

#     df_avis = pd.read_csv(fichier_sortie, sep=";", encoding="utf-8-sig")

#     compte_avis = df_avis.groupby("adresse_pharmacie").size()

#     adresses_jamais_traitees = set(df_region["result_label"]) - set(df_avis["adresse_pharmacie"].dropna().unique())
#     adresses_suspectes = set(compte_avis[compte_avis == seuil_suspect].index)

#     adresses_a_reprendre = adresses_jamais_traitees | adresses_suspectes

#     # securite anti-coupure : on retire les adresses deja presentes
#     # dans le fichier de rattrapage lui-meme, qu'il vienne d'une
#     # execution precedente complete ou interrompue en cours de route
#     adresses_deja_rattrapees = set()
#     if fichier_rattrapage and os.path.exists(fichier_rattrapage):
#         df_rattrapage_existant = pd.read_csv(fichier_rattrapage, sep=";", encoding="utf-8-sig")
#         adresses_deja_rattrapees = set(df_rattrapage_existant["adresse_pharmacie"].dropna().unique())
#         adresses_a_reprendre = adresses_a_reprendre - adresses_deja_rattrapees

#     print(f"Pharmacies jamais traitees (absentes du fichier principal) : {len(adresses_jamais_traitees)}")
#     print(f"Pharmacies avec exactement {seuil_suspect} avis (suspectes) : {len(adresses_suspectes)}")
#     if adresses_deja_rattrapees:
#         print(f"Deja rattrapees lors d'une execution precedente : {len(adresses_deja_rattrapees)} (ignorees)")
#     print(f"Total a reprendre maintenant : {len(adresses_a_reprendre)}")

#     df_a_reprendre = df_region[df_region["result_label"].isin(adresses_a_reprendre)].copy()
#     return df_a_reprendre


# async def rattraper_pharmacies_manquantes(fichier_region, fichier_sortie, fichier_rattrapage, seuil_suspect=10, nb_pages_parallele=5, taille_sous_lot=25):
#     df_a_reprendre = identifier_pharmacies_a_reprendre(fichier_region, fichier_sortie, fichier_rattrapage, seuil_suspect)

#     if df_a_reprendre.empty:
#         print("Rien a rattraper.")
#         return

#     print(f"\nRelance du scraping sur {len(df_a_reprendre)} pharmacies, "
#           f"resultats sauvegardes dans : {fichier_rattrapage}")

#     sous_lots = decouper_en_sous_lots(df_a_reprendre, taille_sous_lot)

#     for num_sous_lot, sous_lot in enumerate(sous_lots, start=1):
#         print(f"Sous-lot {num_sous_lot}/{len(sous_lots)} ({len(sous_lot)} pharmacies)")

#         for tentative in range(2):
#             try:
#                 await scraper_lot_pharmacies(
#                     sous_lot,
#                     chemin_csv_sortie=fichier_rattrapage,
#                     nb_pages_parallele=nb_pages_parallele,
#                     mode_ajout=True,
#                 )
#                 break
#             except Exception as erreur_navigateur:
#                 print(f"Erreur navigateur sur ce sous-lot : {erreur_navigateur}")
#                 if tentative == 0:
#                     print("Nouvelle tentative...")
#                     await asyncio.sleep(5)
#                 else:
#                     print("Sous-lot abandonne apres 2 tentatives, on continue avec le suivant.")

In [46]:
# # Rattrapage

# await rattraper_pharmacies_manquantes(
#     fichier_region="pharmacies_par_region/pharmacies_auvergne_rhone_alpes.csv",
#     fichier_sortie="avis_auvergne_rhone_alpes.csv",
#     fichier_rattrapage="avis_auvergne_rhone_alpes_rattrapage.csv",
#     seuil_suspect=10,
#     nb_pages_parallele=4)

In [47]:
# print(os.path.exists("avis_auvergne_rhone_alpes_rattrapage.csv"))
# df_check = pd.read_csv("avis_auvergne_rhone_alpes_rattrapage.csv", sep=";", encoding="utf-8-sig")
# print(df_check.shape)
# print("Pharmacies distinctes :", df_check["adresse_pharmacie"].nunique())

In [48]:
# # Vérification du scraping total 
# # fichier principal  + rattrapage.

# def verifier_resultat_complet(fichier_region, fichier_sortie, fichier_rattrapage, seuil_suspect=10):
#     df_region = pd.read_csv(fichier_region, sep=";")
#     total_pharmacies = df_region["result_label"].nunique()

#     # on combine les avis du fichier principal et du rattrapage
#     morceaux = []
#     if os.path.exists(fichier_sortie):
#         morceaux.append(pd.read_csv(fichier_sortie, sep=";", encoding="utf-8-sig"))
#     if os.path.exists(fichier_rattrapage):
#         morceaux.append(pd.read_csv(fichier_rattrapage, sep=";", encoding="utf-8-sig"))

#     if not morceaux:
#         print("Aucun fichier trouve.")
#         return

#     df_avis = pd.concat(morceaux, ignore_index=True)

#     # pour chaque adresse, on garde le MEILLEUR nombre d'avis trouve
#     # (utile si le rattrapage a ameliore un resultat du fichier principal)
#     compte_par_adresse = df_avis.groupby("adresse_pharmacie").size()

#     adresses_scrapees = set(compte_par_adresse.index)
#     adresses_manquantes = set(df_region["result_label"]) - adresses_scrapees
#     adresses_a_dix = compte_par_adresse[compte_par_adresse == seuil_suspect]

#     print(f"Nombre total de pharmacies dans la region : {total_pharmacies}")
#     print(f"Pharmacies scrapees avec succes              : {len(adresses_scrapees)}")
#     print(f"Pharmacies manquantes (jamais scrapees)       : {len(adresses_manquantes)}")
#     print(f"Pharmacies avec exactement {seuil_suspect} avis (suspectes) : {len(adresses_a_dix)}")
#     print(f"Total d'avis collectes                        : {len(df_avis)}")


# verifier_resultat_complet(
#     fichier_region="pharmacies_par_region/pharmacies_auvergne_rhone_alpes.csv",
#     fichier_sortie="avis_auvergne_rhone_alpes.csv",
#     fichier_rattrapage="avis_auvergne_rhone_alpes_rattrapage.csv",
#     seuil_suspect=10,
# )

In [51]:
# # concat des 2 fichiers de notre région Auvergne 
# df_principal = pd.read_csv("avis_auvergne_rhone_alpes.csv", sep=";", encoding="utf-8-sig")
# df_rattrapage = pd.read_csv("avis_auvergne_rhone_alpes_rattrapage.csv", sep=";", encoding="utf-8-sig")

# df_final = pd.concat([df_principal, df_rattrapage], ignore_index=True)

# print("Principal :", len(df_principal), "lignes")
# print("Rattrapage :", len(df_rattrapage), "lignes")
# print("Total fusionne :", len(df_final), "lignes")

# df_final.to_csv("avis_auvergne_rhone_alpes_FINAL.csv", sep=";", index=False, encoding="utf-8-sig")

Principal : 104274 lignes
Rattrapage : 39386 lignes
Total fusionne : 143660 lignes


In [61]:
df_region = pd.read_csv("pharmacies_par_region/pharmacies_auvergne_rhone_alpes.csv", sep=";")
df_avis = pd.read_csv("avis_auvergne_rhone_alpes_FINAL.csv", sep=";", encoding="utf-8-sig")

total_pharmacies = df_region["result_label"].nunique()
compte_par_adresse = df_avis.groupby("adresse_pharmacie").size()
adresses_scrapees = set(compte_par_adresse.index)
nb_scrapees = len(adresses_scrapees)
nb_avec_0_avis = total_pharmacies - nb_scrapees
nb_avec_10_avis = (compte_par_adresse == 10).sum()

print("Total pharmacies :", total_pharmacies)
print(f"Pharmacies scrapees : {nb_scrapees} / {total_pharmacies}")
print("Pharmacies avec 0 avis :", nb_avec_0_avis)
print("Pharmacies avec exactement 10 avis :", nb_avec_10_avis)

Total pharmacies : 2345
Pharmacies scrapees : 2272 / 2345
Pharmacies avec 0 avis : 73
Pharmacies avec exactement 10 avis : 10


Remarque : si y a du temps, supprimer les noms/prénoms mentionnés dans les commentaires (> éthique + RGPD)

## Region 2 : Bourgogne-Franche-Comte

In [53]:
await scraper_une_region(
    fichier_region="pharmacies_par_region/pharmacies_bourgogne_franche_comte.csv",
    fichier_sortie="avis_bourgogne_franche_comte.csv",
    nb_pages_parallele=4,
)

Region : pharmacies_par_region/pharmacies_bourgogne_franche_comte.csv
Nombre total de communes : 489
[1/489] Commune : Sens (10 pharmacies)
Pharmacie Pharmacie Ragot : 5 avis recuperes
Pharmacie Pharmacie Fourre Sangare : 21 avis recuperes
Pharmacie Pharmacie de l'Europe : 37 avis recuperes
Pharmacie Pharmacie d'Abraham : 94 avis recuperes
Pharmacie Pharmacie des Chaillots : 98 avis recuperes
Pharmacie Pharmacie Voulx : 61 avis recuperes
Pharmacie Pharmacie Lafayette de la cathédrale : 108 avis recuperes
Pharmacie Pharmacie Pierre de Coubertin : 81 avis recuperes
Pharmacie Pharmacie Croquet Vicherat : 52 avis recuperes
Pharmacie Pharmacie Champbertrand : 38 avis recuperes
Lot traite. Fichier sauvegarde : avis_bourgogne_franche_comte.csv
[2/489] Commune : Nevers (15 pharmacies)
2 pharmacie(s) ignoree(s) car sans nom renseigne.
Pharmacie Pharmacie Saint-Benin : 0 avis recuperes
Pharmacie Pharmacie Beaume Bechtel : 12 avis recuperes
Pharmacie Pharmacie Delgutte : 1 avis recuperes
Pharmaci

In [62]:
df_region = pd.read_csv("pharmacies_par_region/pharmacies_bourgogne_franche_comte.csv", sep=";")
df_avis = pd.read_csv("avis_bourgogne_franche_comte.csv", sep=";", encoding="utf-8-sig")

total_pharmacies = df_region["result_label"].nunique()
compte_par_adresse = df_avis.groupby("adresse_pharmacie").size()
adresses_scrapees = set(compte_par_adresse.index)
nb_scrapees = len(adresses_scrapees)
nb_avec_0_avis = total_pharmacies - nb_scrapees
nb_avec_10_avis = (compte_par_adresse == 10).sum()

print("Total pharmacies :", total_pharmacies)
print(f"Pharmacies scrapees : {nb_scrapees} / {total_pharmacies}")
print("Pharmacies avec 0 avis :", nb_avec_0_avis)
print("Pharmacies avec 10 avis :", nb_avec_10_avis)

Total pharmacies : 952
Pharmacies scrapees : 849 / 952
Pharmacies avec 0 avis : 103
Pharmacies avec 10 avis : 12


## Region 3 : Centre-Val de Loire

In [64]:
await scraper_une_region(
    fichier_region="pharmacies_par_region/pharmacies_centre_val_de_loire.csv",
    fichier_sortie="avis_centre_val_de_loire.csv",
    nb_pages_parallele=4,
)

Region : pharmacies_par_region/pharmacies_centre_val_de_loire.csv
Nombre total de communes : 363
[1/363] Commune : Orléans (25 pharmacies)
1 pharmacie(s) ignoree(s) car sans nom renseigne.
Pharmacie Pharmacie Henri : 25 avis recuperes
Pharmacie Pharmacie des Blossières : 37 avis recuperes
Pharmacie Pharmacie Val de Loire : 82 avis recuperes
Pharmacie Pharmacie Bigarre : 47 avis recuperes
Pharmacie Pharmacie des Tourelles : 57 avis recuperes
Pharmacie Pharmacie de France : 74 avis recuperes
Pharmacie Pharmacie Fontvieille Marliere : 40 avis recuperes
Pharmacie Pharmacie Dunois : 77 avis recuperes
Pharmacie Pharmacie Émile Zola : 43 avis recuperes


Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed\nCall log:\n  - waiting for locator("div[data-review-id]").nth(406).locator(".wiI7pd").first\n')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed
Call log:
  - waiting for locator("div[data-review-id]").nth(406).locator(".wiI7pd").first



Pharmacie Pharmacie de la Nouvelle Orléans : 26 avis recuperes
Pharmacie Pharmacie Centrale : 147 avis recuperes
Pharmacie Pharmacie du Théâtre : 0 avis recuperes
Pharmacie Pharmacie Chaborel : 147 avis recuperes
Pharmacie Pharmacie de l'Argonne : 32 avis recuperes
Pharmacie Pharmacie de la Croix Morin (Lafayette) : 306 avis recuperes
Pharmacie Pharmacie de la Croix Fleury : 73 avis recuperes
Pharmacie Pharmacie Saint-Jean M Plateau : 53 avis recuperes
Pharmacie Pharmacie Gambetta : 46 avis recuperes
Pharmacie Pharmacie de la Poste : 57 avis recuperes
Pharmacie Pharmacie de la Bolière : 143 avis recuperes
Pharmacie Pharmacie de l'Indien : 104 avis recuperes
Pharmacie Pharmacie du Rond Point : 131 avis recuperes
Pharmacie Grande Pharmacie du Châtelet : 103 avis recuperes
Pharmacie Pharmacie de la Cigogne : 583 avis recuperes
Lot traite. Fichier sauvegarde : avis_centre_val_de_loire.csv
[2/363] Commune : Bourges (21 pharmacies)
1 pharmacie(s) ignoree(s) car sans nom renseigne.
Pharmacie 

In [65]:
df_region = pd.read_csv("pharmacies_par_region/pharmacies_centre_val_de_loire.csv", sep=";")
df_avis = pd.read_csv("avis_centre_val_de_loire.csv", sep=";", encoding="utf-8-sig")

total_pharmacies = df_region["result_label"].nunique()
compte_par_adresse = df_avis.groupby("adresse_pharmacie").size()
adresses_scrapees = set(compte_par_adresse.index)
nb_scrapees = len(adresses_scrapees)
nb_avec_0_avis = total_pharmacies - nb_scrapees
nb_avec_10_avis = (compte_par_adresse == 10).sum()

print("Total pharmacies :", total_pharmacies)
print(f"Pharmacies scrapees : {nb_scrapees} / {total_pharmacies}")
print("Pharmacies avec 0 avis :", nb_avec_0_avis)
print("Pharmacies avec exactement 10 avis :", nb_avec_10_avis)

Total pharmacies : 698
Pharmacies scrapees : 599 / 698
Pharmacies avec 0 avis : 99
Pharmacies avec exactement 10 avis : 9


## Region 4 : Bretagne

In [54]:
await scraper_une_region(
    fichier_region="pharmacies_par_region/pharmacies_bretagne.csv",
    fichier_sortie="avis_bretagne.csv",
    nb_pages_parallele=5,
)

Region : pharmacies_par_region/pharmacies_bretagne.csv
Nombre total de communes : 493
[1/493] Commune : Séné (3 pharmacies)
Pharmacie Pharmacie des Voiles Rouges : 28 avis recuperes
Pharmacie Pharmacie Latouche : 29 avis recuperes
Pharmacie Pharmacie du Poulfanc : 98 avis recuperes
Lot traite. Fichier sauvegarde : avis_bretagne.csv
[2/493] Commune : Hennebont (5 pharmacies)
Pharmacie Pharmacie des ramparts : 0 avis recuperes
Pharmacie Pharmacie de la Maison Rouge : 17 avis recuperes
Pharmacie Pharmacie Le Ray - Manceron : 26 avis recuperes
Pharmacie Pharmacie Kergroix : 37 avis recuperes
Pharmacie Pharmacie de la Gardeloupe : 69 avis recuperes
Lot traite. Fichier sauvegarde : avis_bretagne.csv
[3/493] Commune : Saint-Jacques-de-la-Lande (4 pharmacies)
Pharmacie Pharmacie de la Croix Verte : 24 avis recuperes
Pharmacie Pharmacie Redondo : 29 avis recuperes
Pharmacie Pharmacie Saint-Jacques : 31 avis recuperes
Pharmacie Pharmacie Morinais : 115 avis recuperes
Lot traite. Fichier sauvegar

Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed


Pharmacie Pharmacie de l'Espace Saint-Lambert : 55 avis recuperes
Pharmacie Pharmacie Balzac : 35 avis recuperes
Pharmacie Pharmacie de Gouédic : 59 avis recuperes
Pharmacie Pharmacie Léonard de Vinci : 91 avis recuperes
Pharmacie Pharmacie de l'Université : 49 avis recuperes
Pharmacie Pharmacie des villages : 118 avis recuperes
Pharmacie Pharmacie de Robien : 34 avis recuperes
Lot traite. Fichier sauvegarde : avis_bretagne.csv
[6/493] Commune : Moëlan-sur-Mer (3 pharmacies)
  Resultat suspect (10 avis) pour Pharmacie Chardevel et Lecoq, nouvelle tentative...
Pharmacie Pharmacie du Centre : 26 avis recuperes
Pharmacie Pharmacie Principale : 26 avis recuperes
Pharmacie Pharmacie Chardevel et Lecoq : 10 avis recuperes
Lot traite. Fichier sauvegarde : avis_bretagne.csv
[7/493] Commune : Taupont (1 pharmacies)
Pharmacie Pharmacie LEGAVRE : 17 avis recuperes
Lot traite. Fichier sauvegarde : avis_bretagne.csv
[8/493] Commune : Quéven (2 pharmacies)
Pharmacie Pharmacie Charnal : 29 avis recup

In [63]:
df_region = pd.read_csv("pharmacies_par_region/pharmacies_bretagne.csv", sep=";")
df_avis = pd.read_csv("avis_bretagne.csv", sep=";", encoding="utf-8-sig")

total_pharmacies = df_region["result_label"].nunique()
compte_par_adresse = df_avis.groupby("adresse_pharmacie").size()
adresses_scrapees = set(compte_par_adresse.index)
nb_scrapees = len(adresses_scrapees)
nb_avec_0_avis = total_pharmacies - nb_scrapees
nb_avec_10_avis = (compte_par_adresse == 10).sum()

print("Total pharmacies :", total_pharmacies)
print(f"Pharmacies scrapees : {nb_scrapees} / {total_pharmacies}")
print("Pharmacies avec 0 avis :", nb_avec_0_avis)
print("Pharmacies avec 10 avis :", nb_avec_10_avis)

Total pharmacies : 952
Pharmacies scrapees : 794 / 952
Pharmacies avec 0 avis : 158
Pharmacies avec 10 avis : 9


## Region 5 : Corse

In [66]:
await scraper_une_region(
    fichier_region="pharmacies_par_region/pharmacies_corse.csv",
    fichier_sortie="avis_corse.csv",
    nb_pages_parallele=4,
)

Region : pharmacies_par_region/pharmacies_corse.csv
Nombre total de communes : 53
[1/53] Commune : Bastia (20 pharmacies)
2 pharmacie(s) ignoree(s) car sans nom renseigne.
Pharmacie Pharmacie du Fango : 6 avis recuperes
  Resultat suspect (10 avis) pour Pharmacie de l'Opéra, nouvelle tentative...
Pharmacie Pharmacie Ricci-Luciani : 15 avis recuperes
Pharmacie Pharmacie Dussol : 24 avis recuperes
Pharmacie Pharmacie Luccioni : 17 avis recuperes
Pharmacie Aprium Pharmacie du Palais : 17 avis recuperes
Pharmacie Pharmacie de l'Opéra : 10 avis recuperes
Pharmacie Pharmacie Notre-Dame de Lourdes : 9 avis recuperes
Pharmacie Pharmacie de la Place d'Armes : 41 avis recuperes
Pharmacie Gianvity : 9 avis recuperes
Pharmacie Pharmacie St Pierre : 71 avis recuperes
Pharmacie Pharmacie Santa Maria : 15 avis recuperes
Pharmacie Pharmacie Aurore : 7 avis recuperes
Pharmacie Pharmacie Saint-Roch : 9 avis recuperes
Pharmacie Pharmacie Balesi : 92 avis recuperes
Pharmacie Pharmacie Dominici : 126 avis 

In [67]:
df_region = pd.read_csv("pharmacies_par_region/pharmacies_corse.csv", sep=";")
df_avis = pd.read_csv("avis_corse.csv", sep=";", encoding="utf-8-sig")

total_pharmacies = df_region["result_label"].nunique()
compte_par_adresse = df_avis.groupby("adresse_pharmacie").size()
adresses_scrapees = set(compte_par_adresse.index)
nb_scrapees = len(adresses_scrapees)
nb_avec_0_avis = total_pharmacies - nb_scrapees
nb_avec_10_avis = (compte_par_adresse == 10).sum()

print("Total pharmacies :", total_pharmacies)
print(f"Pharmacies scrapees : {nb_scrapees} / {total_pharmacies}")
print("Pharmacies avec 0 avis :", nb_avec_0_avis)
print("Pharmacies avec exactement 10 avis :", nb_avec_10_avis)

Total pharmacies : 112
Pharmacies scrapees : 82 / 112
Pharmacies avec 0 avis : 30
Pharmacies avec exactement 10 avis : 1


# Autres régions : 
## Region 6 : Ile de France

In [ ]:
# await scraper_une_region(
#     fichier_region="pharmacies_par_region/pharmacies_ile_de_france.csv",
#     fichier_sortie="avis_ile_de_france.csv",
#     nb_pages_parallele=4,
# )

## Région 7 : Normandie

In [ ]:
# await scraper_une_region(
#     fichier_region="pharmacies_par_region/pharmacies_normandie.csv",
#     fichier_sortie="avis_normandie.csv",
#     nb_pages_parallele=4,
# )

## Région 8 : Hauts de France

In [ ]:
# await scraper_une_region(
#     fichier_region="pharmacies_par_region/pharmacies_hauts_de_france.csv",
#     fichier_sortie="avis_hauts_de_france.csv",
#     nb_pages_parallele=4,
# )

## Région 9 : Grand Est

In [ ]:
# await scraper_une_region(
#     fichier_region="pharmacies_par_region/pharmacies_grand_est.csv",
#     fichier_sortie="avis_grand_est.csv",
#     nb_pages_parallele=3,
# )

## Région 10 : Pays de la Loire

In [ ]:
# await scraper_une_region(
#     fichier_region="pharmacies_par_region/pharmacies_pays_de_la_loire.csv",
#     fichier_sortie="avis_pays_de_la_loire.csv",
#     nb_pages_parallele=4,
# )

## Région 11 : Nouvelle Aquitaine

In [ ]:
# await scraper_une_region(
#     fichier_region="pharmacies_par_region/pharmacies_nouvelle_aquitaine.csv",
#     fichier_sortie="avis_nouvelle_aquitaine.csv",
#     nb_pages_parallele=4,
# )

## Région 12 : Occitanie

In [ ]:
# await scraper_une_region(
#     fichier_region="pharmacies_par_region/pharmacies_occitanie.csv",
#     fichier_sortie="avis_occitanie.csv",
#     nb_pages_parallele=4,
# )

## Région 13 : Provence Alpes Côtes d'Azur

In [ ]:
# await scraper_une_region(
#     fichier_region="pharmacies_par_region/pharmacies_provence_alpes_cote_d_azur.csv",
#     fichier_sortie="avis_provence_alpes_cote_d_azur.csv",
#     nb_pages_parallele=4,
# )